In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm


def crop_and_filter_dataset(
        root_dir,
        output_dir,
        splits=['train', 'val', 'test'],
        patch_size=256,
        min_building_pixels=15
):
    """
    Нарезает пары снимков LEVIR-CD на патчи 256x256,
    бинаризует маски и отсеивает пустые патчи без зданий.
    """
    for split in splits:
        img_dir_A = os.path.join(root_dir, split, 'A')
        img_dir_B = os.path.join(root_dir, split, 'B')
        label_dir = os.path.join(root_dir, split, 'label')

        # Папки для сохранения результатов
        out_A = os.path.join(output_dir, split, 'A')
        out_B = os.path.join(output_dir, split, 'B')
        out_L = os.path.join(output_dir, split, 'label')

        for p in [out_A, out_B, out_L]:
            os.makedirs(p, exist_ok=True)

        if not os.path.exists(img_dir_A):
            print(f"Предупреждение: папка {img_dir_A} не найдена, пропускаем сплит.")
            continue

        file_list = sorted([f for f in os.listdir(img_dir_A) if f.endswith(('.png', '.jpg'))])

        saved_count = 0
        skipped_count = 0

        print(f"\nОбработка сплита [{split}]: найдено {len(file_list)} исходных пар...")

        for fname in tqdm(file_list):
            imgA = cv2.imread(os.path.join(img_dir_A, fname))
            imgB = cv2.imread(os.path.join(img_dir_B, fname))
            mask = cv2.imread(os.path.join(label_dir, fname), cv2.IMREAD_GRAYSCALE)

            if imgA is None or imgB is None or mask is None:
                continue

            h, w = mask.shape

            for i in range(0, h, patch_size):
                for j in range(0, w, patch_size):
                    patch_mask = mask[i:i + patch_size, j:j + patch_size]

                    # 1. Бинаризация маски (убираем артефакты 156 и 254: всё > 127 ставим в 255)
                    patch_mask = np.where(patch_mask > 127, 255, 0).astype(np.uint8)

                    # 2. Фильтрация пустых патчей (исключаем черные картинки без изменений)
                    building_pixels = np.count_nonzero(patch_mask)
                    if building_pixels < min_building_pixels:
                        skipped_count += 1
                        continue

                    # Вырезаем патчи для снимков A и B
                    patch_A = imgA[i:i + patch_size, j:j + patch_size]
                    patch_B = imgB[i:i + patch_size, j:j + patch_size]

                    # Формируем уникальное имя файла для патча
                    base_name = os.path.splitext(fname)[0]
                    patch_name = f"{base_name}_r{i}_c{j}.png"

                    # Сохраняем результаты
                    cv2.imwrite(os.path.join(out_A, patch_name), patch_A)
                    cv2.imwrite(os.path.join(out_B, patch_name), patch_B)
                    cv2.imwrite(os.path.join(out_L, patch_name), patch_mask)

                    saved_count += 1

        print(f"Сплит [{split}] завершен. Сохранено патчей: {saved_count}, отсеяно пустых: {skipped_count}")


if __name__ == "__main__":

    LEVIR_DIR = r"D:\KURS4\ComputerVision\LEVIR-CD"
    OUTPUT_DATASET = "LEVIR_processed_5K"

    crop_and_filter_dataset(
        root_dir=LEVIR_DIR,
        output_dir=OUTPUT_DATASET,
        splits=['train', 'val', 'test'],
        patch_size=256,
        min_building_pixels=15  # Порог: если пикселей зданий меньше 15, патч отбрасывается
    )